In [1]:
import pandas as pd
import json
from pathlib import Path

In [2]:
file = Path("../data/raw_data/lineups")
sample_file = next(file.iterdir())

In [3]:
with open(sample_file, 'r') as f:
    data = json.load(f)

In [4]:
lineups = data[0]["lineup"]
lineup_data = pd.DataFrame(lineups)
lineup_data.head(50)

,player_id,player_name,player_nickname,jersey_number,country,cards,positions
0,3287,Nathaniel Chalobah,None,94,"{'id': 68, 'name': 'England'}",[],"[{'position_id': 15, 'position': 'Left Center ..."
1,3949,Manolo Gabbiadini,None,23,"{'id': 112, 'name': 'Italy'}",[],[]
2,5497,Gonzalo Gerardo Higuaín,Gonzalo Higuaín,9,"{'id': 11, 'name': 'Argentina'}",[],"[{'position_id': 23, 'position': 'Center Forwa..."
3,5630,Dries Mertens,None,14,"{'id': 22, 'name': 'Belgium'}",[],"[{'position_id': 21, 'position': 'Left Wing', ..."
4,5675,Kalidou Koulibaly,None,26,"{'id': 202, 'name': 'Senegal'}",[],"[{'position_id': 5, 'position': 'Left Center B..."
5,6754,David López Silva,David López,19,"{'id': 214, 'name': 'Spain'}",[],"[{'position_id': 13, 'position': 'Right Center..."
6,6952,Alberto Grassi,None,88,"{'id': 112, 'name': 'Italy'}",[],[]
7,7012,Vasco Regini,None,18,"{'id': 112, 'name': 'Italy'}",[],[]
8,7024,Jorge Luiz Frello Filho,Jorginho,8,"{'id': 112, 'name': 'Italy'}",[],"[{'position_id': 10, 'position': 'Center Defen..."
9,7025,Marek Hamšík,None,17,"{'id': 207, 'name': 'Slovakia'}",[],"[{'position_id': 15, 'position': 'Left Center ..."


In [5]:
lineup_data["country"][0]

{'id': 68, 'name': 'England'}

In [6]:
lineup_data[lineup_data["positions"].notna()].iloc[0]["positions"]

[{'position_id': 15,
  'position': 'Left Center Midfield',
  'from': '85:38',
  'to': None,
  'from_period': 2,
  'to_period': None,
  'start_reason': 'Substitution - On (Tactical)',
  'end_reason': 'Final Whistle'}]

In [7]:
sample_position = lineup_data[lineup_data["positions"].notna()].iloc[0]["positions"]

In [8]:
sample_position[0].keys()

dict_keys(['position_id', 'position', 'from', 'to', 'from_period', 'to_period', 'start_reason', 'end_reason'])

In [9]:
with open("../data/raw_data/matches/2/27.json", "r") as f:
    match_data = json.load(f)

print(type(match_data))
print(type(match_data[0]))
print(match_data[0].keys())

<class 'list'>
<class 'dict'>
dict_keys(['match_id', 'match_date', 'kick_off', 'competition', 'season', 'home_team', 'away_team', 'home_score', 'away_score', 'match_status', 'match_status_360', 'last_updated', 'last_updated_360', 'metadata', 'match_week', 'competition_stage', 'stadium', 'referee'])


In [10]:
print(match_data[0]["competition"])
print(match_data[0]["season"])
print(match_data[0]["home_team"])
print(match_data[0]["away_team"])

{'competition_id': 2, 'country_name': 'England', 'competition_name': 'Premier League'}
{'season_id': 27, 'season_name': '2015/2016'}
{'home_team_id': 22, 'home_team_name': 'Leicester City', 'home_team_gender': 'male', 'home_team_group': None, 'country': {'id': 68, 'name': 'England'}, 'managers': [{'id': 60, 'name': 'Claudio Ranieri', 'nickname': None, 'dob': '1951-10-20', 'country': {'id': 112, 'name': 'Italy'}}]}
{'away_team_id': 28, 'away_team_name': 'AFC Bournemouth', 'away_team_gender': 'male', 'away_team_group': None, 'country': {'id': 68, 'name': 'England'}, 'managers': [{'id': 38, 'name': 'Eddie Howe', 'nickname': None, 'dob': '1977-11-29', 'country': {'id': 68, 'name': 'England'}}]}


In [11]:
match_data[0]["match_week"]

20

In [12]:
competitions = list(match_data[0]["competition"].keys())
seasons = list(match_data[0]["season"].keys())

print(competitions)
print(seasons)

['competition_id', 'country_name', 'competition_name']
['season_id', 'season_name']


In [13]:
manager_rows = []

raw_managers = (match_data[0]["home_team"]["managers"] + match_data[0]["away_team"]["managers"])   

for manager in raw_managers:
     manager_row = {
    'manager_id': manager['id'],
    'manager_name': manager['name'],
    'date_of_birth': manager['dob'],
    'country_id': manager["country"]["id"],
    'country_name': manager["country"]["name"]}

     manager_rows.append(manager_row)

In [14]:
manager_rows

[{'manager_id': 60,
  'manager_name': 'Claudio Ranieri',
  'date_of_birth': '1951-10-20',
  'country_id': 112,
  'country_name': 'Italy'},
 {'manager_id': 38,
  'manager_name': 'Eddie Howe',
  'date_of_birth': '1977-11-29',
  'country_id': 68,
  'country_name': 'England'}]

In [15]:
team_managers = [{"match_id": match_data[0]["match_id"], "team_id": match_data[0]["home_team"]["home_team_id"], "manager_id": match_data[0]["home_team"]["managers"][0]["id"]}, {"match_id": match_data[0]["match_id"], "team_id": match_data[0]["away_team"]["away_team_id"], "manager_id": match_data[0]["away_team"]["managers"][0]["id"]}]
team_managers

[{'match_id': 3754058, 'team_id': 22, 'manager_id': 60},
 {'match_id': 3754058, 'team_id': 28, 'manager_id': 38}]

In [16]:
team_manager_rows = []

for team_key, team_id_key in [("home_team", "home_team_id"),("away_team", "away_team_id")]:
    team = match_data[0][team_key]
    for manager in team.get("managers", []):
        team_manager_rows.append(
            {   "match_id": match_data[0]["match_id"],
                "team_id": team[team_id_key],
                "manager_id": manager["id"],
            }
        )

In [17]:
match_data[0]

{'match_id': 3754058,
 'match_date': '2016-01-02',
 'kick_off': '16:00:00.000',
 'competition': {'competition_id': 2,
  'country_name': 'England',
  'competition_name': 'Premier League'},
 'season': {'season_id': 27, 'season_name': '2015/2016'},
 'home_team': {'home_team_id': 22,
  'home_team_name': 'Leicester City',
  'home_team_gender': 'male',
  'home_team_group': None,
  'country': {'id': 68, 'name': 'England'},
  'managers': [{'id': 60,
    'name': 'Claudio Ranieri',
    'nickname': None,
    'dob': '1951-10-20',
    'country': {'id': 112, 'name': 'Italy'}}]},
 'away_team': {'away_team_id': 28,
  'away_team_name': 'AFC Bournemouth',
  'away_team_gender': 'male',
  'away_team_group': None,
  'country': {'id': 68, 'name': 'England'},
  'managers': [{'id': 38,
    'name': 'Eddie Howe',
    'nickname': None,
    'dob': '1977-11-29',
    'country': {'id': 68, 'name': 'England'}}]},
 'home_score': 0,
 'away_score': 0,
 'match_status': 'available',
 'match_status_360': 'processing',
 'la

In [18]:
match_list = []
for i in range(len(match_data)):
    match_dict = {}
    match_dict["match_id"] = match_data[i]["match_id"]
    match_dict["match_date"] = match_data[i]["match_date"]
    match_dict["match_week"] = match_data[i]["match_week"]
    match_dict["home_score"] = match_data[i]["home_score"]
    match_dict["away_score"] = match_data[i]["away_score"]
    match_dict["home_team_id"] = match_data[i]["home_team"]["home_team_id"]
    match_dict["away_team_id"] = match_data[i]["away_team"]["away_team_id"]
    match_dict["competition_id"] = match_data[i]["competition"]["competition_id"]
    match_dict["season_id"] = match_data[i]["season"]["season_id"]
    match_dict["competition_stage"] = match_data[i]["competition_stage"]["id"]
    match_list.append(match_dict)
match_list[:10]

[{'match_id': 3754058,
  'match_date': '2016-01-02',
  'match_week': 20,
  'home_score': 0,
  'away_score': 0,
  'home_team_id': 22,
  'away_team_id': 28,
  'competition_id': 2,
  'season_id': 27,
  'competition_stage': 1},
 {'match_id': 3754245,
  'match_date': '2015-10-17',
  'match_week': 9,
  'home_score': 1,
  'away_score': 0,
  'home_team_id': 27,
  'away_team_id': 41,
  'competition_id': 2,
  'season_id': 27,
  'competition_stage': 1},
 {'match_id': 3754136,
  'match_date': '2015-12-19',
  'match_week': 17,
  'home_score': 1,
  'away_score': 1,
  'home_team_id': 37,
  'away_team_id': 59,
  'competition_id': 2,
  'season_id': 27,
  'competition_stage': 1},
 {'match_id': 3754037,
  'match_date': '2016-04-30',
  'match_week': 36,
  'home_score': 2,
  'away_score': 1,
  'home_team_id': 29,
  'away_team_id': 28,
  'competition_id': 2,
  'season_id': 27,
  'competition_stage': 1},
 {'match_id': 3754039,
  'match_date': '2016-02-13',
  'match_week': 26,
  'home_score': 1,
  'away_score

In [19]:
match_df = pd.DataFrame(match_list)
match_df.head()

,match_id,match_date,match_week,home_score,away_score,home_team_id,away_team_id,competition_id,season_id,competition_stage
0,3754058,2016-01-02,20,0,0,22,28,2,27,1
1,3754245,2015-10-17,9,1,0,27,41,2,27,1
2,3754136,2015-12-19,17,1,1,37,59,2,27,1
3,3754037,2016-04-30,36,2,1,29,28,2,27,1
4,3754039,2016-02-13,26,1,2,31,23,2,27,1


In [20]:
match_df.columns.to_list()

['match_id',
 'match_date',
 'match_week',
 'home_score',
 'away_score',
 'home_team_id',
 'away_team_id',
 'competition_id',
 'season_id',
 'competition_stage']

In [21]:
teams_list = []
for i in range(len(match_data)):
    teams_dict = {}
    teams_dict["team_id"] = match_data[i]["home_team"]["home_team_id"]
    teams_dict["team_id"] = match_data[i]["away_team"]["away_team_id"]
    teams_dict["team_name"] = match_data[i]["home_team"]["home_team_name"]
    teams_dict["team_name"] = match_data[i]["away_team"]["away_team_name"]
    teams_list.append(teams_dict)
teams_list[:10]

[{'team_id': 28, 'team_name': 'AFC Bournemouth'},
 {'team_id': 41, 'team_name': 'Sunderland'},
 {'team_id': 59, 'team_name': 'Aston Villa'},
 {'team_id': 28, 'team_name': 'AFC Bournemouth'},
 {'team_id': 23, 'team_name': 'Watford'},
 {'team_id': 59, 'team_name': 'Aston Villa'},
 {'team_id': 24, 'team_name': 'Liverpool'},
 {'team_id': 28, 'team_name': 'AFC Bournemouth'},
 {'team_id': 36, 'team_name': 'Manchester City'},
 {'team_id': 29, 'team_name': 'Everton'}]

In [22]:
teams_df = pd.DataFrame(teams_list)
teams_df

,team_id,team_name
0,28,AFC Bournemouth
1,41,Sunderland
2,59,Aston Villa
3,28,AFC Bournemouth
4,23,Watford
...,...,...
375,28,AFC Bournemouth
376,27,West Bromwich Albion
377,40,West Ham United
378,59,Aston Villa


In [23]:
teams_df = teams_df.drop_duplicates(subset=["team_id"]).reset_index(drop=True)
teams_df

,team_id,team_name
0,28,AFC Bournemouth
1,41,Sunderland
2,59,Aston Villa
3,23,Watford
4,24,Liverpool
5,36,Manchester City
6,29,Everton
7,56,Norwich City
8,31,Crystal Palace
9,38,Tottenham Hotspur


In [24]:
manager_list = []

for i in range(len(match_data)):
     manager_dict = {
    'manager_id': match_data[i]['home_team']['managers'][0]['id'],
    'manager_id': match_data[i]['away_team']['managers'][0]['id'],
    'manager_name': match_data[i]['home_team']['managers'][0]['name'],
    'manager_name': match_data[i]['away_team']['managers'][0]['name'],
    'nickname': match_data[i]['home_team']['managers'][0]['nickname'],
    'nickname': match_data[i]['away_team']['managers'][0]['nickname'],
    'date_of_birth': match_data[i]['home_team']['managers'][0]['dob'],
    'date_of_birth': match_data[i]['away_team']['managers'][0]['dob'],
    'country_id': match_data[i]['home_team']['managers'][0]['country']['id'],
    'country_id': match_data[i]['away_team']['managers'][0]['country']['id'],
    'country_name': match_data[i]['home_team']['managers'][0]['country']['name'],
    'country_name': match_data[i]['away_team']['managers'][0]['country']['name']}

     manager_list.append(manager_dict)

manager_list[:5]

[{'manager_id': 38,
  'manager_name': 'Eddie Howe',
  'nickname': None,
  'date_of_birth': '1977-11-29',
  'country_id': 68,
  'country_name': 'England'},
 {'manager_id': 561,
  'manager_name': 'Sam Allardyce',
  'nickname': None,
  'date_of_birth': '1954-10-19',
  'country_id': 68,
  'country_name': 'England'},
 {'manager_id': 92,
  'manager_name': 'Rémi Garde',
  'nickname': None,
  'date_of_birth': '1966-04-03',
  'country_id': 78,
  'country_name': 'France'},
 {'manager_id': 38,
  'manager_name': 'Eddie Howe',
  'nickname': None,
  'date_of_birth': '1977-11-29',
  'country_id': 68,
  'country_name': 'England'},
 {'manager_id': 236,
  'manager_name': 'Enrique Sánchez Flores',
  'nickname': 'Quique Sánchez Flores',
  'date_of_birth': '1965-02-05',
  'country_id': 214,
  'country_name': 'Spain'}]

In [25]:
managers_df = pd.DataFrame(manager_list)
managers_df

,manager_id,manager_name,nickname,date_of_birth,country_id,country_name
0,38,Eddie Howe,None,1977-11-29,68,England
1,561,Sam Allardyce,None,1954-10-19,68,England
2,92,Rémi Garde,None,1966-04-03,78,France
3,38,Eddie Howe,None,1977-11-29,68,England
4,236,Enrique Sánchez Flores,Quique Sánchez Flores,1965-02-05,214,Spain
...,...,...,...,...,...,...
375,38,Eddie Howe,None,1977-11-29,68,England
376,300,Tony Pulis,None,1958-01-16,249,Wales
377,150,Slaven Bilić,None,1968-09-11,56,Croatia
378,5003,Tim Sherwood,None,1969-02-06,68,England


In [26]:
managers_df = managers_df.drop_duplicates(subset=["manager_id"]).reset_index(drop=True)
managers_df

,manager_id,manager_name,nickname,date_of_birth,country_id,country_name
0,38,Eddie Howe,None,1977-11-29,68,England
1,561,Sam Allardyce,None,1954-10-19,68,England
2,92,Rémi Garde,None,1966-04-03,78,France
3,236,Enrique Sánchez Flores,Quique Sánchez Flores,1965-02-05,214,Spain
4,5020,Eric Black,None,1963-10-01,201,Scotland
5,94,Jürgen Klopp,None,1967-06-16,85,Germany
6,733,Manuel Luis Pellegrini Ripamonti,Manuel Pellegrini,1953-09-16,45,Chile
7,263,Roberto Martínez Montoliú,Roberto Martínez,1973-07-13,214,Spain
8,41,Alex Neil,None,1981-06-09,201,Scotland
9,382,Alan Pardew,None,1961-07-18,68,England


In [27]:
team_managers_list = []

for i in range(len(match_data)):
    home_team = match_data[i]["home_team"]
    away_team = match_data[i]["away_team"]

    for manager in home_team["managers"]:
        team_managers_list.append({"match_id": match_data[i]["match_id"], 
                           "team_id":  home_team["home_team_id"], 
                           "manager_id": manager["id"]})
    for manager in away_team["managers"]:
        team_managers_list.append({"match_id": match_data[i]["match_id"], 
                           "team_id":  away_team["away_team_id"], 
                           "manager_id": manager["id"]})
team_managers_list[:3]

[{'match_id': 3754058, 'team_id': 22, 'manager_id': 60},
 {'match_id': 3754058, 'team_id': 28, 'manager_id': 38},
 {'match_id': 3754245, 'team_id': 27, 'manager_id': 300}]

In [28]:
data[0]

{'team_id': 227,
 'team_name': 'Napoli',
 'lineup': [{'player_id': 3287,
   'player_name': 'Nathaniel Chalobah',
   'player_nickname': None,
   'jersey_number': 94,
   'country': {'id': 68, 'name': 'England'},
   'cards': [],
   'positions': [{'position_id': 15,
     'position': 'Left Center Midfield',
     'from': '85:38',
     'to': None,
     'from_period': 2,
     'to_period': None,
     'start_reason': 'Substitution - On (Tactical)',
     'end_reason': 'Final Whistle'}]},
  {'player_id': 3949,
   'player_name': 'Manolo Gabbiadini',
   'player_nickname': None,
   'jersey_number': 23,
   'country': {'id': 112, 'name': 'Italy'},
   'cards': [],
   'positions': []},
  {'player_id': 5497,
   'player_name': 'Gonzalo Gerardo Higuaín',
   'player_nickname': 'Gonzalo Higuaín',
   'jersey_number': 9,
   'country': {'id': 11, 'name': 'Argentina'},
   'cards': [],
   'positions': [{'position_id': 23,
     'position': 'Center Forward',
     'from': '00:00',
     'to': None,
     'from_period':

In [29]:
data[0]["lineup"][0].keys()

dict_keys(['player_id', 'player_name', 'player_nickname', 'jersey_number', 'country', 'cards', 'positions'])

In [30]:
data[0]["lineup"][0]['country'].keys()

dict_keys(['id', 'name'])

In [31]:
data[0]["lineup"][0]['positions'][0].keys()

dict_keys(['position_id', 'position', 'from', 'to', 'from_period', 'to_period', 'start_reason', 'end_reason'])

In [32]:
with open("../data/raw_data/lineups/7472.json", 'r') as f:
    sample_data = json.load(f)

In [33]:
sample_data[0]['lineup'][11]['cards'][0].keys()

dict_keys(['time', 'card_type', 'reason', 'period'])

In [34]:
data[0]['lineup'][0]['positions'][0]

{'position_id': 15,
 'position': 'Left Center Midfield',
 'from': '85:38',
 'to': None,
 'from_period': 2,
 'to_period': None,
 'start_reason': 'Substitution - On (Tactical)',
 'end_reason': 'Final Whistle'}

In [35]:
sample_data[0]['lineup'][11]['cards'][0]

{'time': '34:35',
 'card_type': 'Yellow Card',
 'reason': 'Foul Committed',
 'period': 1}

In [36]:
len(data)

2

In [37]:
data[1].keys()

dict_keys(['team_id', 'team_name', 'lineup'])

In [38]:
players_list = []
lineups_list = []
countries_list = []

for item in file.iterdir():    
    with open(item, 'r') as f:
        data = json.load(f)

    for team in data:
        for player in team['lineup']:
            player_dict = {
            'player_id': player['player_id'],
            'player_name': player['player_name'],
            'country_id': player.get('country', {}).get('id')
            }
            lineup_dict = {
            'match_id': int(item.stem),
            'team_id': team['team_id'],
            'player_id': player['player_id'],
            'jersey_number': player['jersey_number']
            }
            countries_dict = {
            'country_id': player.get('country', {}).get('id'),
            'country_name': player.get('country', {}).get('name')
            }
            players_list.append(player_dict)
            lineups_list.append(lineup_dict)
            countries_list.append(countries_dict)
print(players_list[0])
print(lineups_list[0])
print(countries_list[0])

{'player_id': 3287, 'player_name': 'Nathaniel Chalobah', 'country_id': 68}
{'match_id': 3879769, 'team_id': 227, 'player_id': 3287, 'jersey_number': 94}
{'country_id': 68, 'country_name': 'England'}


In [39]:
players_df = pd.DataFrame(players_list)
lineups_df = pd.DataFrame(lineups_list)
countries_df = pd.DataFrame(countries_list)

print(len(players_df))
print(len(lineups_df))
print(len(countries_df))

131901
131901
131901


In [40]:
players_df = players_df.drop_duplicates(subset=['player_id']).reset_index(drop=True)
countries_df = countries_df.drop_duplicates(subset=['country_id']).reset_index(drop=True)
print(len(players_df))
print(len(countries_df))

10803
142


In [41]:
positions_list = []
cards_list = []
for item in file.iterdir():    
    with open(item, 'r') as f:
        data = json.load(f)

    for team in data:
        for player in team['lineup']:
            for position in player["positions"]:
                position_dict = {
                    'match_id': int(item.stem),
                    'player_id': player['player_id'],
                    'position_id': position['position_id'],
                    'position_name': position['position'],
                    'from_time': position['from'],
                    'to_time': position['to'],
                    'from_period': position['from_period'],
                    'to_period': position['to_period'],
                    'start_reason': position['start_reason'],
                    'end_reason': position['end_reason']
                }
                positions_list.append(position_dict)
            for card in player["cards"]:
                cards_dict = {
                    'match_id': int(item.stem),
                    'player_id': player['player_id'],
                    'card_time': card['time'],
                    'card_type': card['card_type'],
                    'reason': card['reason'],
                    'period': card['period']
                }
                cards_list.append(cards_dict)

positions_list[0]
cards_list[0]

{'match_id': 3879769,
 'player_id': 7049,
 'card_time': '10:15',
 'card_type': 'Yellow Card',
 'reason': 'Foul Committed',
 'period': 1}

In [42]:
positions_df = pd.DataFrame(positions_list)
positions_df.head()

,match_id,player_id,position_id,position_name,from_time,to_time,from_period,to_period,start_reason,end_reason
0,3879769,3287,15,Left Center Midfield,85:38,None,2,NaN,Substitution - On (Tactical),Final Whistle
1,3879769,5497,23,Center Forward,00:00,None,1,NaN,Starting XI,Final Whistle
2,3879769,5630,21,Left Wing,70:44,None,2,NaN,Substitution - On (Tactical),Final Whistle
3,3879769,5675,5,Left Center Back,00:00,None,1,NaN,Starting XI,Final Whistle
4,3879769,6754,13,Right Center Midfield,00:00,66:45,1,2.0,Starting XI,Substitution - Off (Tactical)


In [43]:
cards_df = pd.DataFrame(cards_list)
cards_df.head()

,match_id,player_id,card_time,card_type,reason,period
0,3879769,7049,10:15,Yellow Card,Foul Committed,1
1,3879769,8877,57:00,Yellow Card,Foul Committed,2
2,3879769,11643,81:41,Yellow Card,Foul Committed,2
3,16157,5211,90:23,Yellow Card,Bad Behaviour,2
4,16157,5470,46:01,Yellow Card,Foul Committed,1


In [44]:
cards_df["card_type"].value_counts(dropna=False)
cards_df["reason"].value_counts(dropna=False)
cards_df["period"].value_counts(dropna=False)

period
2    9052
1    4975
4      52
3      27
5       5
Name: count, dtype: int64

In [45]:
for match in match_data:
    for team_key, team_id_key in [("home_team", "home_team_id"),("away_team", "away_team_id")]:
        team = match[team_key]
        print(team)
        print(match["match_id"])
        # for managers in team['managers']:
        #      print(managers)
        #      print(managers['country']['id'])

{'home_team_id': 22, 'home_team_name': 'Leicester City', 'home_team_gender': 'male', 'home_team_group': None, 'country': {'id': 68, 'name': 'England'}, 'managers': [{'id': 60, 'name': 'Claudio Ranieri', 'nickname': None, 'dob': '1951-10-20', 'country': {'id': 112, 'name': 'Italy'}}]}
3754058
{'away_team_id': 28, 'away_team_name': 'AFC Bournemouth', 'away_team_gender': 'male', 'away_team_group': None, 'country': {'id': 68, 'name': 'England'}, 'managers': [{'id': 38, 'name': 'Eddie Howe', 'nickname': None, 'dob': '1977-11-29', 'country': {'id': 68, 'name': 'England'}}]}
3754058
{'home_team_id': 27, 'home_team_name': 'West Bromwich Albion', 'home_team_gender': 'male', 'home_team_group': None, 'country': {'id': 68, 'name': 'England'}, 'managers': [{'id': 300, 'name': 'Tony Pulis', 'nickname': None, 'dob': '1958-01-16', 'country': {'id': 249, 'name': 'Wales'}}]}
3754245
{'away_team_id': 41, 'away_team_name': 'Sunderland', 'away_team_gender': 'male', 'away_team_group': None, 'country': {'id'

In [ ]:
players_df.head()